# Part B — Machine Learning Models
## ACE6313 Machine Learning — Food Waste (SDG 12)

**Author:** Student 2 (Machine Learning Models)

Run Part A first to generate `clean_food_waste.csv`. This notebook trains 6 algorithms,
tunes hyperparameters, and compares performance.


## 1. Imports & Load Clean Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay)

df = pd.read_csv("../data/clean_food_waste.csv")
print("Shape:", df.shape)
df.head()

## 2. Train/Test Split

In [ ]:
X = df.drop(columns=["Confidence_encoded"])
y = df["Confidence_encoded"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Class distribution in y:")
print(y.value_counts().sort_index())

## 3. Train 6 Models (at least 5 required)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree":       DecisionTreeClassifier(random_state=42),
    "Random Forest":       RandomForestClassifier(random_state=42),
    "SVM":                 SVC(),
    "kNN":                 KNeighborsClassifier(),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
}

baseline = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    baseline[name] = {
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall":    recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1":        f1_score(y_test, pred, average="weighted", zero_division=0),
    }

baseline_df = pd.DataFrame(baseline).T.round(3)
print("Baseline performance (before tuning):")
baseline_df

## 4. Hyperparameter Tuning
Tune 3 models with GridSearchCV to show improvement.

In [ ]:
# --- Tune Random Forest ---
rf_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [None, 5, 10, 20],
    "min_samples_split": [2, 5, 10],
}
rf_search = GridSearchCV(RandomForestClassifier(random_state=42),
                         rf_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
rf_search.fit(X_train, y_train)
print("RF best params:", rf_search.best_params_)
print("RF best CV F1:", round(rf_search.best_score_, 3))

In [ ]:
# --- Tune SVM ---
svm_grid = {
    "C": [0.1, 1, 10, 100],
    "kernel": ["linear", "rbf"],
    "gamma": ["scale", "auto"],
}
svm_search = GridSearchCV(SVC(), svm_grid, cv=5, scoring="f1_weighted", n_jobs=-1)
svm_search.fit(X_train, y_train)
print("SVM best params:", svm_search.best_params_)
print("SVM best CV F1:", round(svm_search.best_score_, 3))

In [ ]:
# --- Tune kNN ---
knn_grid = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"],
}
knn_search = GridSearchCV(KNeighborsClassifier(), knn_grid, cv=5,
                          scoring="f1_weighted", n_jobs=-1)
knn_search.fit(X_train, y_train)
print("kNN best params:", knn_search.best_params_)
print("kNN best CV F1:", round(knn_search.best_score_, 3))

## 5. Evaluate Tuned Models

In [ ]:
tuned_models = {
    "Random Forest (tuned)": rf_search.best_estimator_,
    "SVM (tuned)":           svm_search.best_estimator_,
    "kNN (tuned)":           knn_search.best_estimator_,
}

tuned = {}
for name, model in tuned_models.items():
    pred = model.predict(X_test)
    tuned[name] = {
        "Accuracy":  accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, average="weighted", zero_division=0),
        "Recall":    recall_score(y_test, pred, average="weighted", zero_division=0),
        "F1":        f1_score(y_test, pred, average="weighted", zero_division=0),
    }

tuned_df = pd.DataFrame(tuned).T.round(3)
print("Tuned performance:")
tuned_df

In [ ]:
# Detailed report for the best model
best_name = tuned_df["F1"].idxmax()
best_model = tuned_models[best_name]
pred = best_model.predict(X_test)
print(f"Best model: {best_name}\n")
print(classification_report(y_test, pred, zero_division=0))

## 6. Compare All Models

In [ ]:
# Combine baseline + tuned into one comparison table
all_results = pd.concat([baseline_df, tuned_df]).round(3)
print(all_results)
all_results.to_csv("../report/model_results.csv")

In [ ]:
# Bar chart of F1-scores
all_results["F1"].sort_values().plot(kind="barh", figsize=(8,6), color="steelblue")
plt.title("Model F1-Score Comparison")
plt.xlabel("Weighted F1-Score")
plt.tight_layout()
plt.savefig("../report/model_comparison.png", dpi=150)
plt.show()

In [ ]:
# Confusion matrix for the best model
cm = confusion_matrix(y_test, pred)
disp = ConfusionMatrixDisplay(cm, display_labels=["Very Low","Low","Medium","High"])
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig("../report/confusion_matrix.png", dpi=150)
plt.show()

---
### Summary (write this up in the report)
- Trained 6 algorithms; tuned 3 with GridSearchCV (5-fold, F1-weighted).
- Best model: see output above. Compare baseline vs tuned to show the gain.
- **Discuss the class imbalance** (Very Low = 130 vs High = 10). Weighted F1 accounts for it;
  mention SMOTE / class weighting as future work.
